# rPPG — 4. tur: sıfırdan temiz eğitim + telefon benzeri bozulmalar (Kaggle)

* **Sıfırdan:** 1. tur modelinden devam etmek yerine PURE ön-eğitimli FactorizePhys'ten başlar (öğrenme oranı
  3e-4, cosine, ~160 epoch). 2. ve 3. turda, eğitimi bitmiş bir modeli yeniden ısıtınca doğrulama hatası bozuldu.
* **Temiz etiketler:** 3. turdaki temizlik (MCD yan kamera zaman kayması, kötü etiketli bölümler, UBFC-Phys hizalama).
* **Telefon benzeri bozulmalar** (`--phone-aug`): el titremesi, düşük çözünürlük/sıkıştırma bulanıklığı, renk sıcaklığı,
  loş ışık + sensör gürültüsü. Eğitim verisi laboratuvar webcam'i, uygulama ise telefonda çalışıyor.
* Doğrulama hatası artık kaynak/kamera bazında da yazılır.

Sonda 1. ve 3. tur modelleri aynı 28 test videosunda yan yana ölçülür.

**Veri:** MCD-rPPG (590 kişi), PURE (10 kişi × 6 hareket), UBFC-rPPG (34 denek; 1. turda çöken çözme düzeltildi) ve
**UBFC-Phys** (56 kişi × 3 görev: dinlenme, konuşma, aritmetik stres; Hugging Face'te, erişim gerektirir).

**Ayarlar (Settings menüsü):** Accelerator **GPU T4 x2**, **Turn on internet**.

**UBFC-Phys için (önerilir):**
1. Hugging Face'te giriş yapıp https://huggingface.co/datasets/jjuik2014/UBFC-Phys-all sayfasında koşulları kabul edin.
2. huggingface.co → Settings → Access Tokens → "Read" anahtarı oluşturun.
3. Kaggle'da **Add-ons → Secrets → Add**: Label `HF_TOKEN`, Value: anahtar; bu not defterine bağlayın.

Anahtar yoksa UBFC-Phys atlanır, diğer veriyle eğitim yine yapılır.

**Çalıştırma:** **Save Version → Save & Run All (Commit)**; ≈11 saat. Bir önceki sürümün Secrets bağlantısı
(HF_TOKEN) kopyalanan not defterinde de açık olmalı. Bitince **Output** sekmesinden `cikti` klasörünü
indirin.

In [ ]:
# 1) Ortam, kod ve rPPG-Toolbox
import os, subprocess, shutil, sys, time, glob, json
from datetime import datetime, timedelta

T0 = datetime.now()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert shutil.which("nvidia-smi"), "GPU yok: Settings -> Accelerator: GPU T4 x2"
assert subprocess.run(["curl", "-sI", "https://github.com"], capture_output=True).returncode == 0, \
    "İnternet kapalı: Settings -> Turn on internet"
print("ffmpeg:", shutil.which("ffmpeg"))

REPO = "/kaggle/working/rppg"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Nisagwn/rppg.git", REPO], check=True)
TB = os.path.join(REPO, "external", "rPPG-Toolbox")
if not os.path.exists(TB):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ubicomplab/rPPG-Toolbox.git", TB], check=True)
os.chdir(REPO)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN var: UBFC-Phys denenecek")
except Exception:
    print("!" * 70)
    print("UYARI: HF_TOKEN yok, UBFC-Phys atlanacak. Add-ons -> Secrets -> HF_TOKEN'ın yanındaki "
          "kutuyu işaretleyin ve not defterini yeniden başlatın.")
    print("!" * 70)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# 2) Klasörler, süre, başlangıç modeli
DATA, RAW = "/tmp/dl", "/tmp/dl_raw"
OUT = "/kaggle/working/cikti"
LOGS = os.path.join(OUT, "gunlukler")
for d in (DATA, RAW, OUT, LOGS):
    os.makedirs(d, exist_ok=True)

TRAIN_UNTIL = T0 + timedelta(hours=12 - 1.6)          # Kaggle oturum sınırı 12 saat; değerlendirme payı
INIT = os.path.join(REPO, "results/derin_ogrenme/kaggle/cikti/best.pth")   # 1. tur, epoch 84 (yalnızca karşılaştırma)
assert os.path.exists(INIT), INIT
free = shutil.disk_usage("/tmp").free / 1e9
MAX_SEC = 45 if free > 300 else 30
print(f"/tmp boş {free:.0f} GB | MCD/UBFC-Phys süre sınırı {MAX_SEC} s | eğitim en geç {TRAIN_UNTIL:%H:%M}")

In [ ]:
# 3) Veri hazırlama (arka planda, indir-işle-sil)
#    A: MCD-rPPG 600 kişi   B: test kümesi -> UBFC 34 -> PURE 60   C: UBFC-Phys (erişim varsa)
PY = sys.executable
env = dict(os.environ, PYTHONUNBUFFERED="1")

def launch(name, cmd):
    return subprocess.Popen(["bash", "-c", cmd], cwd=REPO, env=env,
                            stdout=open(os.path.join(LOGS, name + ".log"), "a"), stderr=subprocess.STDOUT)

P = f"{PY} -u scripts/dl_prepare.py --out {DATA} --raw {RAW}"
jobs = {
    "hazirla_mcd": launch("hazirla_mcd", f"{P} --mcd 600 --workers 2 --max-sec {MAX_SEC}"),
    "hazirla_test_ubfc_pure": launch("hazirla_test_ubfc_pure",
        f"{P} --test-download --workers 2; {P} --ubfc 34 --workers 1; {P} --pure 60 --workers 1"),
    "hazirla_ubfcphys": launch("hazirla_ubfcphys", f"{P} --ubfcphys 48 --workers 1 --max-sec 120"),
}
print("veri hazırlama başladı")

In [ ]:
# 4) Eğitim: 300 video hazır olunca başlar; yeni gelenler her epoch başında eklenir
SUBS = ("mcd", "ubfc", "pure", "ubfcphys")
def counts():
    return {s: len(glob.glob(os.path.join(DATA, s, "*.npz"))) for s in SUBS}

while sum(counts().values()) < 300 and any(j.poll() is None for j in jobs.values()):
    print(time.strftime("%H:%M"), counts(), flush=True)
    time.sleep(120)

train = subprocess.Popen(
    [PY, "-u", "scripts/dl_train.py", "--data", DATA, "--out", OUT, "--epochs", "160",
     "--steps", "500", "--lr", "3e-4", "--phone-aug", "--deadline", TRAIN_UNTIL.strftime("%Y-%m-%d %H:%M")],
    cwd=REPO, env=env, stdout=open(os.path.join(LOGS, "egitim.log"), "a"), stderr=subprocess.STDOUT)
print("eğitim başladı")

In [ ]:
# 5) İzleme: 10 dakikada bir özet
def tail(path, n):
    try:
        return open(path, encoding="utf-8", errors="replace").read().splitlines()[-n:]
    except OSError:
        return []

while train.poll() is None:
    ep = [l for l in tail(os.path.join(LOGS, "egitim.log"), 400) if l.startswith("epoch ")]
    print(f"{time.strftime('%H:%M')} | {counts()} | disk {shutil.disk_usage('/tmp').free/1e9:.0f} GB | "
          f"{ep[-1] if ep else 'ilk epoch sürüyor'}", flush=True)
    time.sleep(600)
print("\n".join(tail(os.path.join(LOGS, "egitim.log"), 15)))

In [ ]:
# 6) Veri kalitesi özeti (eğitimde kullanılan filtre; kalite.csv çıktıya kopyalanır)
r = subprocess.run([PY, "scripts/dl_quality.py", "--data", DATA, "--workers", "4",
                    "--sources", "mcd", "ubfc", "pure", "ubfcphys"], cwd=REPO, capture_output=True, text=True, env=env)
print(r.stdout[-3000:], r.stderr[-2000:])
if os.path.exists(os.path.join(DATA, "kalite.csv")):
    shutil.copy(os.path.join(DATA, "kalite.csv"), OUT)

In [ ]:
# 7) Değerlendirme: aynı 28 test videosu
for _ in range(60):
    if len(glob.glob(os.path.join(DATA, "test", "*.npz"))) >= 28 or jobs["hazirla_test_ubfc_pure"].poll() is not None:
        break
    time.sleep(60)
for j in jobs.values():
    if j.poll() is None:
        j.terminate()
print("test videoları:", len(glob.glob(os.path.join(DATA, "test", "*.npz"))))
r = subprocess.run([PY, "scripts/dl_evaluate.py", "--model", os.path.join(OUT, "best.pth"),
                    "--test", os.path.join(DATA, "test"), "--data", DATA, "--out", OUT,
                    "--history", os.path.join(OUT, "gecmis.json"),
                    "--compare", f"1. tur (başlangıç)={INIT}",
                    f"3. tur (son epoch)={REPO}/results/derin_ogrenme/kaggle_tur3/last.pth"],
                   cwd=REPO, capture_output=True, text=True, env=env)
print(r.stdout[-6000:], r.stderr[-3000:])

In [ ]:
# 8) Çıktılar
for p in sorted(glob.glob(os.path.join(OUT, "*"))):
    print(f"{os.path.getsize(p)/1e6:8.2f} MB  {p}" if os.path.isfile(p) else f"         {p}/")
shutil.rmtree(REPO, ignore_errors=True)   # çıktıda yalnızca cikti/ kalsın
print("süre:", datetime.now() - T0)